# 2 — Data Preprocessing Pipeline

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Authors:** Eren Acar Başaran, Ahmet Aybars Pektaş  

---

This notebook covers **Work Package 2 — Preprocessing Pipeline** from the project proposal.

**Deliverables:**
- D2.1: Cleaned and Split Dataset ready for modeling
- D2.2: Scikit-Learn Preprocessing Pipeline Script

## 2.1 — Imports & Load Raw Data

Import libraries (Pandas, NumPy, Scikit-Learn). Load the raw dataset from `../data/vehicles.csv` and print the shape. No filtering here — that happens in section 2.3.

In [3]:
# 2.1 — Imports & Load Raw Data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = "../data/vehicles.csv"

# Load raw dataset
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (426880, 26)


## 2.2 — Feature Selection & Column Dropping

Based on the EDA findings, drop columns that are not useful for modeling:
- **Identifiers:** `id`, `url`, `region_url`, `image_url`, `VIN` — not predictive
- **Free-text:** `description` — NLP out of scope
- **Geography:** `lat`, `long`, `region` — `state` kept instead
- **Excessive missing:** `county` (100%), `size` (71.8%), `cylinders` (41.6%) — too many NaN to be useful
- **Temporal:** `posting_date` — not relevant to price

Print the remaining columns and shape after dropping.

In [4]:
# 2.2 — Feature Selection & Column Dropping

# Identifiers — no predictive value
# Free-text / URL fields — not used (text analysis is out of scope)
# Columns with excessive missing data or no data at all (county is 100% NaN)
# Location coordinates — too granular, state is kept instead
# posting_date — not a vehicle feature
# region — redundant with state
# size — over 70% missing in EDA
cols_to_drop = [
    "id", "url", "region_url", "image_url",  # identifiers / URLs
    "description",                             # free-text, out of scope
    "VIN",                                     # identifier
    "county",                                  # 100% NaN
    "lat", "long",                             # coordinates, too granular
    "posting_date",                            # not a vehicle feature
    "region",                                  # redundant with state
    "size",                                    # >70% missing
    "cylinders",                               # 41.6% missing, correlated with manufacturer/model
]

df = df.drop(columns=cols_to_drop, errors="ignore")

# state kept for potential use in future iterations
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping columns: (426880, 13)
Remaining columns: ['price', 'year', 'manufacturer', 'model', 'condition', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']


## 2.3 — Outlier Filtering

Apply hard-bound filtering based on EDA findings:
- Drop rows where `price` is NaN.
- Keep rows where `price` >= $500 **and** `price` <= $100,000 (removes spam listings and unrealistic values).
- Drop rows where `odometer` > 300,000 miles.

Report the number of rows before and after filtering.

In [5]:
# 2.3 — Outlier Filtering
rows_before = len(df)

# Drop rows where price is NaN
df = df.dropna(subset=["price"])

# Price filter: hard min $500, hard max $100,000
# Keeps legitimate vehicles while removing $0/$1 spam and unrealistic listings
df = df[(df["price"] >= 500) & (df["price"] <= 100_000)]

# Odometer: drop rows above 300,000 miles
df = df[df["odometer"].fillna(0) <= 300_000]

rows_after = len(df)
print(f"Before filtering: {rows_before:,} rows")
print(f"After filtering:  {rows_after:,} rows")
print(f"Rows removed:     {rows_before - rows_after:,}")

Before filtering: 426,880 rows
After filtering:  381,427 rows
Rows removed:     45,453


## 2.4 — Train-Test Split

**!! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!**

- Separate features (`X`) and target (`y = price`).
- Perform an 80/20 train-test split using `train_test_split` with a fixed `random_state` for reproducibility.
- Report the shape of train and test sets.

In [6]:
# 2.4 — Train-Test Split
# !! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (305141, 12)
X_test:  (76286, 12)
y_train: (305141,)
y_test:  (76286,)


## 2.5 — Missing Value Imputation

Build imputation strategies (fit strictly on training data only):
- **Numerical features** (`year`, `odometer`): Impute with median values.
- **Categorical features** (`manufacturer`, `condition`, `fuel`, `transmission`, `drive`, `type`, etc.): Impute with the mode (most frequent value).

Use Scikit-Learn's `SimpleImputer` within the pipeline so that imputation statistics are learned from training data and applied consistently to test data.

In [8]:
# 2.5 — Missing Value Imputation
# All group statistics are learned from X_train ONLY, stored as dicts, then applied to both sets.

print("Missing values BEFORE imputation:")
print(X_train.isna().sum())
print()

# ============================================================
# YEAR — group by odometer bins (3000-mile bins), median year
# ============================================================
BIN_SIZE = 3000
X_train["_odo_bin"] = (X_train["odometer"] // BIN_SIZE) * BIN_SIZE
year_by_odo_bin = X_train.groupby("_odo_bin")["year"].median().to_dict()
overall_median_year = X_train["year"].median()

def impute_year(row, mapping, fallback):
    if pd.isna(row["year"]):
        odo_bin = (row["odometer"] // BIN_SIZE) * BIN_SIZE if pd.notna(row["odometer"]) else None
        return mapping.get(odo_bin, fallback)
    return row["year"]

X_train["year"] = X_train.apply(impute_year, axis=1, mapping=year_by_odo_bin, fallback=overall_median_year)
X_train.drop(columns=["_odo_bin"], inplace=True)

X_test["_odo_bin"] = (X_test["odometer"] // BIN_SIZE) * BIN_SIZE
X_test["year"] = X_test.apply(impute_year, axis=1, mapping=year_by_odo_bin, fallback=overall_median_year)
X_test.drop(columns=["_odo_bin"], inplace=True)

# ============================================================
# ODOMETER — group by year, median odometer
# ============================================================
odo_by_year = X_train.groupby("year")["odometer"].median().to_dict()
overall_median_odo = X_train["odometer"].median()

def impute_odometer(row, mapping, fallback):
    if pd.isna(row["odometer"]):
        return mapping.get(row["year"], fallback)
    return row["odometer"]

X_train["odometer"] = X_train.apply(impute_odometer, axis=1, mapping=odo_by_year, fallback=overall_median_odo)
X_test["odometer"] = X_test.apply(impute_odometer, axis=1, mapping=odo_by_year, fallback=overall_median_odo)

# ============================================================
# CONDITION — 50K-mile odometer bins, most frequent condition per bin
# Bins: 0-50K, 50K-100K, 100K-150K, 150K-200K, 200K+
# Fallback: 'good'
# ============================================================
COND_BIN_EDGES = [0, 50_000, 100_000, 150_000, 200_000, np.inf]
COND_BIN_LABELS = ["0-50K", "50K-100K", "100K-150K", "150K-200K", "200K+"]

X_train["_cond_bin"] = pd.cut(X_train["odometer"], bins=COND_BIN_EDGES, labels=COND_BIN_LABELS, right=False)
cond_by_bin = (
    X_train.dropna(subset=["condition"])
    .groupby("_cond_bin", observed=True)["condition"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

def impute_condition(row, mapping):
    if pd.notna(row["condition"]):
        return row["condition"]
    cond_bin = row.get("_cond_bin")
    if pd.notna(cond_bin) and cond_bin in mapping:
        return mapping[cond_bin]
    return "good"

X_train["condition"] = X_train.apply(impute_condition, axis=1, mapping=cond_by_bin)
X_train.drop(columns=["_cond_bin"], inplace=True)

X_test["_cond_bin"] = pd.cut(X_test["odometer"], bins=COND_BIN_EDGES, labels=COND_BIN_LABELS, right=False)
X_test["condition"] = X_test.apply(impute_condition, axis=1, mapping=cond_by_bin)
X_test.drop(columns=["_cond_bin"], inplace=True)

# ============================================================
# MANUFACTURER — group by model, most frequent manufacturer
# Fallback: 'unknown' if model is also missing
# ============================================================
mfr_by_model = (
    X_train.dropna(subset=["manufacturer", "model"])
    .groupby("model")["manufacturer"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

def impute_manufacturer(row, mapping):
    if pd.isna(row["manufacturer"]):
        if pd.notna(row["model"]):
            return mapping.get(row["model"], "unknown")
        return "unknown"
    return row["manufacturer"]

X_train["manufacturer"] = X_train.apply(impute_manufacturer, axis=1, mapping=mfr_by_model)
X_test["manufacturer"] = X_test.apply(impute_manufacturer, axis=1, mapping=mfr_by_model)

# ============================================================
# MODEL — group by manufacturer, most frequent model
# Fallback: 'unknown'
# ============================================================
model_by_mfr = (
    X_train.dropna(subset=["model", "manufacturer"])
    .groupby("manufacturer")["model"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

def impute_model(row, mapping):
    if pd.isna(row["model"]):
        return mapping.get(row["manufacturer"], "unknown")
    return row["model"]

X_train["model"] = X_train.apply(impute_model, axis=1, mapping=model_by_mfr)
X_test["model"] = X_test.apply(impute_model, axis=1, mapping=model_by_mfr)

# ============================================================
# ALL OTHER CATEGORICALS — group by model → manufacturer → overall
# Most frequent value at each level, cascading fallback
# ============================================================
other_cat_cols = ["fuel", "transmission", "drive", "type", "title_status", "paint_color", "state"]

# Build mappings: {col: {model: mode_val}}, {col: {manufacturer: mode_val}}, {col: overall_mode}
mode_by_model = {}
mode_by_mfr = {}
mode_overall = {}

for col in other_cat_cols:
    valid = X_train.dropna(subset=[col])
    mode_by_model[col] = valid.dropna(subset=["model"]).groupby("model")[col].agg(lambda x: x.mode().iloc[0]).to_dict()
    mode_by_mfr[col] = valid.dropna(subset=["manufacturer"]).groupby("manufacturer")[col].agg(lambda x: x.mode().iloc[0]).to_dict()
    mode_overall[col] = X_train[col].mode().iloc[0] if not X_train[col].mode().empty else "unknown"

def impute_other_cat(row, col):
    if pd.isna(row[col]):
        # Try model-level mode
        if pd.notna(row["model"]) and row["model"] in mode_by_model[col]:
            return mode_by_model[col][row["model"]]
        # Try manufacturer-level mode
        if pd.notna(row["manufacturer"]) and row["manufacturer"] in mode_by_mfr[col]:
            return mode_by_mfr[col][row["manufacturer"]]
        # Overall mode
        return mode_overall[col]
    return row[col]

for col in other_cat_cols:
    X_train[col] = X_train.apply(lambda row, c=col: impute_other_cat(row, c), axis=1)
    X_test[col] = X_test.apply(lambda row, c=col: impute_other_cat(row, c), axis=1)

# ============================================================
# Verify: zero missing values remaining
# ============================================================
print("Missing values AFTER imputation:")
print(X_train.isna().sum())
print()
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

Missing values BEFORE imputation:
year                 0
manufacturer     11764
model             3520
condition       115288
fuel              2071
odometer             0
title_status      5615
transmission      1456
drive            92707
type             65584
paint_color      88273
state                0
dtype: int64

Missing values AFTER imputation:
year            0
manufacturer    0
model           0
condition       0
fuel            0
odometer        0
title_status    0
transmission    0
drive           0
type            0
paint_color     0
state           0
dtype: int64

X_train shape: (305141, 12)
X_test shape:  (76286, 12)


## 2.6 — Feature Encoding

Encode categorical features for model consumption:
- **Low-cardinality features** (e.g., `fuel`, `transmission`, `condition`, `drive`, `type`, `title_status`): Use One-Hot Encoding or Ordinal Encoding as appropriate.
- **High-cardinality features** (e.g., `manufacturer`, `model`, `state`): Apply **Target Encoding** to avoid dimensionality explosion.

Wrap all encoding in Scikit-Learn `ColumnTransformer` to keep the pipeline clean.

## 2.7 — Feature Scaling (Optional)

- Apply `StandardScaler` to numerical features for the Linear Regression model (tree-based models do not require scaling, but it won't hurt).
- This step can be toggled on/off per model within the final pipeline.

## 2.8 — Assemble the Scikit-Learn Pipeline

Build the full `sklearn.pipeline.Pipeline` object that chains:
1. `ColumnTransformer` (imputation + encoding for numerical and categorical columns)
2. Optional scaler
3. Placeholder for the estimator (to be swapped per model notebook)

Save/export the preprocessing pipeline so the model notebooks can import it directly.

## 2.9 — Verification & Sanity Checks

- Transform the training data through the pipeline and verify the output shape.
- Check for any remaining NaN values.
- Print a sample of transformed feature names.
- Confirm no test data was used during fitting.